In [1]:
import pandas as pd
import streamlit as st

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from sklearn.model_selection import train_test_split   # splits data into train & test
from sklearn.preprocessing import LabelEncoder          # converts text labels to numbers
from sklearn.linear_model import LogisticRegression     # ML Model 1
from sklearn.ensemble import RandomForestClassifier     # ML Model 2
from sklearn.ensemble import GradientBoostingClassifier # ML Model 3
from sklearn.metrics import accuracy_score              # measures how accurate our model is
from sklearn.metrics import classification_report       # detailed accuracy breakdown
from sklearn.metrics import confusion_matrix     

In [4]:
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## load the dataset

In [5]:
df = pd.read_csv("D:\Dataset-Mental-Disorders.csv")

In [6]:
df.head()

,Patient Number,Sadness,Euphoric,Exhausted,Sleep dissorder,Mood Swing,Suicidal thoughts,Anorxia,Authority Respect,Try-Explanation,Aggressive Response,Ignore & Move-On,Nervous Break-down,Admit Mistakes,Overthinking,Sexual Activity,Concentration,Optimisim,Expert Diagnose
0,Patiant-01,Usually,Seldom,Sometimes,Sometimes,YES,YES,NO,NO,YES,NO,NO,YES,YES,YES,3 From 10,3 From 10,4 From 10,Bipolar Type-2
1,Patiant-02,Usually,Seldom,Usually,Sometimes,NO,YES,NO,NO,NO,NO,NO,NO,NO,NO,4 From 10,2 From 10,5 From 10,Depression
2,Patiant-03,Sometimes,Most-Often,Sometimes,Sometimes,YES,NO,NO,NO,YES,YES,NO,YES,YES,NO,6 From 10,5 From 10,7 From 10,Bipolar Type-1
3,Patiant-04,Usually,Seldom,Usually,Most-Often,YES,YES,YES,NO,YES,NO,NO,NO,NO,NO,3 From 10,2 From 10,2 From 10,Bipolar Type-2
4,Patiant-05,Usually,Usually,Sometimes,Sometimes,NO,NO,NO,NO,NO,NO,NO,YES,YES,YES,5 From 10,5 From 10,6 From 10,Normal


In [7]:
df.shape

(120, 19)

## understanding what inside

In [8]:
# exploratory data analysis
print("="*50)
print("BASIC INFORMATION ABOUT OUR DATASET")
print("="*50)

BASIC INFORMATION ABOUT OUR DATASET


In [9]:
df.dtypes

Patient Number         object
Sadness                object
Euphoric               object
Exhausted              object
Sleep dissorder        object
Mood Swing             object
Suicidal thoughts      object
Anorxia                object
Authority Respect      object
Try-Explanation        object
Aggressive Response    object
Ignore & Move-On       object
Nervous Break-down     object
Admit Mistakes         object
Overthinking           object
Sexual Activity        object
Concentration          object
Optimisim              object
Expert Diagnose        object
dtype: object

In [10]:
df.isnull().sum()

Patient Number         0
Sadness                0
Euphoric               0
Exhausted              0
Sleep dissorder        0
Mood Swing             0
Suicidal thoughts      0
Anorxia                0
Authority Respect      0
Try-Explanation        0
Aggressive Response    0
Ignore & Move-On       0
Nervous Break-down     0
Admit Mistakes         0
Overthinking           0
Sexual Activity        0
Concentration          0
Optimisim              0
Expert Diagnose        0
dtype: int64

In [11]:
# unique condition in expert diagnose
df['Expert Diagnose'].unique()

array(['Bipolar Type-2', 'Depression', 'Bipolar Type-1', 'Normal'],
      dtype=object)

In [12]:
# how many patients per condition
df['Expert Diagnose'].value_counts()

Expert Diagnose
Bipolar Type-2    31
Depression        31
Normal            30
Bipolar Type-1    28
Name: count, dtype: int64

In [13]:
# Let's look at the unique values inside each feature column
# This tells us exactly what kind of text we're dealing with
# so we know how to convert them to numbers

print("🔎 Unique values in each feature column:\n")

# We loop through every column EXCEPT 'Patient Number' and 'Expert Diagnose'
# because Patient Number is just an ID (useless)
# and Expert Diagnose is our target (we'll handle it separately)

for col in df.columns:
    if col not in ['Patient Number', 'Expert Diagnose']:
        unique_vals = df[col].unique()
        print(f"  {col}:")
        print(f"     {unique_vals}\n")

🔎 Unique values in each feature column:

  Sadness:
     ['Usually' 'Sometimes' 'Seldom' 'Most-Often']

  Euphoric:
     ['Seldom' 'Most-Often' 'Usually' 'Sometimes']

  Exhausted:
     ['Sometimes' 'Usually' 'Seldom' 'Most-Often']

  Sleep dissorder:
     ['Sometimes' 'Most-Often' 'Usually' 'Seldom']

  Mood Swing:
     ['YES' 'NO']

  Suicidal thoughts:
     ['YES ' 'YES' 'NO']

  Anorxia:
     ['NO' 'YES']

  Authority Respect:
     ['NO' 'YES']

  Try-Explanation:
     ['YES' 'NO']

  Aggressive Response:
     ['NO' 'YES']

  Ignore & Move-On:
     ['NO' 'YES']

  Nervous Break-down:
     ['YES' 'NO']

  Admit Mistakes:
     ['YES' 'NO']

  Overthinking:
     ['YES' 'NO']

  Sexual Activity:
     ['3 From 10' '4 From 10' '6 From 10' '5 From 10' '7 From 10' '8 From 10'
 '9 From 10' '2 From 10' '1 From 10']

  Concentration:
     ['3 From 10' '2 From 10' '5 From 10' '4 From 10' '7 From 10' '6 From 10'
 '1 From 10' '8 From 10']

  Optimisim:
     ['4 From 10' '5 From 10' '7 From 10' '

## drop the useless column

In [14]:
# 'Patient Number' is just a serial number like 1, 2, 3...
# It has NO relation to mental health whatsoever
# If we keep it, the model might accidentally try to "learn" from it
# which would give us wrong results

df = df.drop(columns=['Patient Number'])

print("✅ Dropped 'Patient Number' column")
print(f"New shape: {df.shape}")
# Should now show one less column than before

✅ Dropped 'Patient Number' column
New shape: (120, 18)


## fixing the hidden sapce bug first

In [15]:
# .str.strip() removes any hidden spaces from the beginning or end of text
# "YES " becomes "YES", "  NO" becomes "NO"
# We do this for ALL columns at once to be safe

# This loops through every column and strips whitespace from text values
for col in df.columns:
    if df[col].dtype == 'object':        # only do this for text columns
        df[col] = df[col].str.strip()    # removes leading/trailing spaces

print("✅ Hidden spaces removed from all columns")

# Verify the fix — Suicidal thoughts should now show only YES and NO
print("\nSuicidal thoughts unique values after fix:")
print(df['Suicidal thoughts'].unique())
# Should now show: ['YES' 'NO']  — no more 'YES ' with a space

✅ Hidden spaces removed from all columns

Suicidal thoughts unique values after fix:
['YES' 'NO']


## convert frequency words to numbers

In [16]:
# Type 1 columns use: Seldom / Sometimes / Usually / Most-Often
# We convert them to 0 / 1 / 2 / 3
# This is called "ordinal encoding" — we preserve the ORDER
# (Most-Often is more than Sometimes, so it gets a higher number)

# This is the mapping — a Python dictionary (key → value pairs)
frequency_map = {
    'Seldom':      0,   # least frequent
    'Sometimes':   1,   # occasionally
    'Usually':     2,   # often
    'Most-Often':  3    # almost always
}

# These are all the columns that use the frequency scale
frequency_cols = [
    'Sadness', 'Euphoric', 'Exhausted', 'Sleep dissorder',
    'Try-Explanation', 'Aggressive Response', 'Ignore & Move-On',
    'Nervous Break-down', 'Admit Mistakes', 'Overthinking',
    'Sexual Activity', 'Concentration'
]

# .map() replaces each text value using our dictionary
# e.g. every cell that says 'Seldom' becomes 0
for col in frequency_cols:
    df[col] = df[col].map(frequency_map)

print("✅ Frequency columns converted to numbers")
print("\nSadness column after conversion:")
print(df['Sadness'].unique())
# Should show: [0, 1, 2, 3]

✅ Frequency columns converted to numbers

Sadness column after conversion:
[2 1 0 3]


##  Convert YES/NO columns to numbers

In [17]:
# Type 2 columns use: YES / NO
# YES → 1 (means the symptom IS present)
# NO  → 0 (means the symptom is NOT present)

yes_no_map = {
    'YES': 1,
    'NO':  0
}

yes_no_cols = [
    'Mood Swing', 'Suicidal thoughts', 'Anorxia',
    'Authority Respect'
]

for col in yes_no_cols:
    df[col] = df[col].map(yes_no_map)

print("✅ YES/NO columns converted to numbers")
print("\nMood Swing unique values after conversion:")
print(df['Mood Swing'].unique())
# Should show: [1, 0]

✅ YES/NO columns converted to numbers

Mood Swing unique values after conversion:
[1 0]


##  Fix the Optimism column 

In [18]:
# Optimism has values like "4 From 10", "7 From 10" etc.
# We only need the NUMBER part — the "From 10" is the same for all rows
# so it carries no information

# .str.split(' ') splits the text by spaces → ["4", "From", "10"]
# [0] takes the first element → "4"
# .astype(int) converts the string "4" to the integer 4

df['Optimisim'] = df['Optimisim'].str.split(' ').str[0].astype(int)

print("✅ Optimism column fixed")
print("\nOptimism unique values after fix:")
print(sorted(df['Optimisim'].unique()))
# Should show: [1, 2, 3, 4, 5, 6, 7, 8, 9]

✅ Optimism column fixed

Optimism unique values after fix:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


##  Convert the target column to numbers

In [19]:
# Our target column 'Expert Diagnose' also has text labels
# We need to convert them to numbers for the ML model
# This is called "Label Encoding"

# We'll use LabelEncoder from sklearn — it automatically assigns numbers
le = LabelEncoder()

# fit_transform() does two things:
# 1. "fit"      → learns all unique labels (Bipolar Type-1, Bipolar Type-2, Depression, Normal)
# 2. "transform"→ replaces each label with a number

df['Expert Diagnose'] = le.fit_transform(df['Expert Diagnose'])

print("✅ Target column encoded")
print("\nLabel mapping (name → number):")
for number, name in enumerate(le.classes_):
    print(f"   {number} → {name}")

print("\nTarget column unique values:")
print(df['Expert Diagnose'].unique())

✅ Target column encoded

Label mapping (name → number):
   0 → Bipolar Type-1
   1 → Bipolar Type-2
   2 → Depression
   3 → Normal

Target column unique values:
[1 2 0 3]


## Final check — make sure everything looks clean

In [20]:
print("="*50)
print("FINAL CLEANED DATASET OVERVIEW")
print("="*50)

print(f"\nShape: {df.shape}")

print("\nData types (all should be int or float now except target):")
print(df.dtypes)

print("\nFirst 5 rows of cleaned data:")
print(df.head())

print("\nAny missing values left?")
print(df.isnull().sum().sum())  
# This should print 0 — meaning no missing values anywhere

FINAL CLEANED DATASET OVERVIEW

Shape: (120, 18)

Data types (all should be int or float now except target):
Sadness                  int64
Euphoric                 int64
Exhausted                int64
Sleep dissorder          int64
Mood Swing               int64
Suicidal thoughts        int64
Anorxia                  int64
Authority Respect        int64
Try-Explanation        float64
Aggressive Response    float64
Ignore & Move-On       float64
Nervous Break-down     float64
Admit Mistakes         float64
Overthinking           float64
Sexual Activity        float64
Concentration          float64
Optimisim                int64
Expert Diagnose          int64
dtype: object

First 5 rows of cleaned data:
   Sadness  Euphoric  Exhausted  Sleep dissorder  Mood Swing  \
0        2         0          1                1           1   
1        2         0          2                1           0   
2        1         3          1                1           1   
3        2         0          2 

In [21]:
# Let's check what values are actually in the frequency columns
# AFTER the strip() but BEFORE we ran the map
# We need to reload the data fresh and check carefully

# First let's see which columns have NaN values
print("Columns with missing values:")
print(df.isnull().sum())
print()

# Now let's check the ORIGINAL raw values again
# We'll reload the dataset fresh to see the original values
df_raw = pd.read_csv("D:\Dataset-Mental-Disorders.csv")

# Strip spaces on the raw data
for col in df_raw.columns:
    if df_raw[col].dtype == 'object':
        df_raw[col] = df_raw[col].str.strip()

# Now check every frequency column for unique values
frequency_cols = [
    'Sadness', 'Euphoric', 'Exhausted', 'Sleep dissorder',
    'Try-Explanation', 'Aggressive Response', 'Ignore & Move-On',
    'Nervous Break-down', 'Admit Mistakes', 'Overthinking',
    'Sexual Activity', 'Concentration'
]

print("Raw unique values in each frequency column:")
for col in frequency_cols:
    print(f"\n  {col}: {df_raw[col].unique()}")

Columns with missing values:
Sadness                  0
Euphoric                 0
Exhausted                0
Sleep dissorder          0
Mood Swing               0
Suicidal thoughts        0
Anorxia                  0
Authority Respect        0
Try-Explanation        120
Aggressive Response    120
Ignore & Move-On       120
Nervous Break-down     120
Admit Mistakes         120
Overthinking           120
Sexual Activity        120
Concentration          120
Optimisim                0
Expert Diagnose          0
dtype: int64

Raw unique values in each frequency column:

  Sadness: ['Usually' 'Sometimes' 'Seldom' 'Most-Often']

  Euphoric: ['Seldom' 'Most-Often' 'Usually' 'Sometimes']

  Exhausted: ['Sometimes' 'Usually' 'Seldom' 'Most-Often']

  Sleep dissorder: ['Sometimes' 'Most-Often' 'Usually' 'Seldom']

  Try-Explanation: ['YES' 'NO']

  Aggressive Response: ['NO' 'YES']

  Ignore & Move-On: ['NO' 'YES']

  Nervous Break-down: ['YES' 'NO']

  Admit Mistakes: ['YES' 'NO']

  Overthink

In [22]:
# We reload the original CSV fresh
# This wipes away all our mistakes and gives us a clean slate

df = pd.read_csv("D:\Dataset-Mental-Disorders.csv")

# Step 1 — Strip hidden spaces from ALL text columns immediately
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()

# Step 2 — Drop Patient Number (useless ID column)
df = df.drop(columns=['Patient Number'])

print("✅ Fresh data loaded and spaces stripped")
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")

✅ Fresh data loaded and spaces stripped
Shape: (120, 18)
Missing values: 0


In [23]:
# NOW we know the correct groups after diagnosis
# ONLY these 4 columns use Seldom/Sometimes/Usually/Most-Often

frequency_map = {
    'Seldom':     0,
    'Sometimes':  1,
    'Usually':    2,
    'Most-Often': 3
}

# CORRECT list — only 4 columns, not 12!
frequency_cols = [
    'Sadness',
    'Euphoric',
    'Exhausted',
    'Sleep dissorder'
]

for col in frequency_cols:
    df[col] = df[col].map(frequency_map)

print("✅ Frequency columns converted")
print("\nUnique values check:")
for col in frequency_cols:
    print(f"  {col}: {sorted(df[col].unique())}")
# Every column should show: [0, 1, 2, 3]

✅ Frequency columns converted

Unique values check:
  Sadness: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
  Euphoric: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
  Exhausted: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
  Sleep dissorder: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


In [24]:
yes_no_map = {
    'YES': 1,
    'NO':  0
}

yes_no_cols = [
    'Mood Swing',
    'Suicidal thoughts',
    'Anorxia',
    'Authority Respect'
]

for col in yes_no_cols:
    df[col] = df[col].map(yes_no_map)

print("✅ YES/NO columns converted")
print("\nUnique values check:")
for col in yes_no_cols:
    print(f"  {col}: {df[col].unique()}")
# Every column should show: [0, 1] or [1, 0]

✅ YES/NO columns converted

Unique values check:
  Mood Swing: [1 0]
  Suicidal thoughts: [1 0]
  Anorxia: [0 1]
  Authority Respect: [0 1]


In [25]:
# Let's print unique values for EVERY column in our problem list
# so we know EXACTLY what format each one uses
# No more guessing!

problem_cols = [
    'Try-Explanation',
    'Aggressive Response',
    'Ignore & Move-On',
    'Nervous Break-down',
    'Admit Mistakes',
    'Overthinking',
    'Sexual Activity',
    'Concentration',
    'Optimisim'
]

# Reload raw data fresh to see original values
df_check = pd.read_csv("D:\Dataset-Mental-Disorders.csv")
for col in df_check.columns:
    if df_check[col].dtype == 'object':
        df_check[col] = df_check[col].str.strip()

print("Raw unique values for each column:\n")
for col in problem_cols:
    print(f"  {col}:")
    print(f"     {df_check[col].unique()}")
    print()

Raw unique values for each column:

  Try-Explanation:
     ['YES' 'NO']

  Aggressive Response:
     ['NO' 'YES']

  Ignore & Move-On:
     ['NO' 'YES']

  Nervous Break-down:
     ['YES' 'NO']

  Admit Mistakes:
     ['YES' 'NO']

  Overthinking:
     ['YES' 'NO']

  Sexual Activity:
     ['3 From 10' '4 From 10' '6 From 10' '5 From 10' '7 From 10' '8 From 10'
 '9 From 10' '2 From 10' '1 From 10']

  Concentration:
     ['3 From 10' '2 From 10' '5 From 10' '4 From 10' '7 From 10' '6 From 10'
 '1 From 10' '8 From 10']

  Optimisim:
     ['4 From 10' '5 From 10' '7 From 10' '2 From 10' '6 From 10' '9 From 10'
 '3 From 10' '8 From 10' '1 From 10']



In [26]:
# Convert Expert Diagnose text labels to numbers
le = LabelEncoder()
df['Expert Diagnose'] = le.fit_transform(df['Expert Diagnose'])

print("✅ Target column encoded")
print("\nLabel mapping:")
for number, name in enumerate(le.classes_):
    print(f"   {number} → {name}")

✅ Target column encoded

Label mapping:
   0 → Bipolar Type-1
   1 → Bipolar Type-2
   2 → Depression
   3 → Normal


In [27]:
print("="*50)
print("FINAL CLEANED DATASET")
print("="*50)

print(f"\nShape: {df.shape}")

print("\nData types:")
print(df.dtypes)

print(f"\nTotal missing values: {df.isnull().sum().sum()}")
# MUST be 0

print("\nFirst 3 rows:")
print(df.head(3))

FINAL CLEANED DATASET

Shape: (120, 18)

Data types:
Sadness                 int64
Euphoric                int64
Exhausted               int64
Sleep dissorder         int64
Mood Swing              int64
Suicidal thoughts       int64
Anorxia                 int64
Authority Respect       int64
Try-Explanation        object
Aggressive Response    object
Ignore & Move-On       object
Nervous Break-down     object
Admit Mistakes         object
Overthinking           object
Sexual Activity        object
Concentration          object
Optimisim              object
Expert Diagnose         int64
dtype: object

Total missing values: 0

First 3 rows:
   Sadness  Euphoric  Exhausted  Sleep dissorder  Mood Swing  \
0        2         0          1                1           1   
1        2         0          2                1           0   
2        1         3          1                1           1   

   Suicidal thoughts  Anorxia  Authority Respect Try-Explanation  \
0                  1        

In [28]:
# ============================================================
# COMPLETE RESTART — Load, clean and fix everything correctly
# ============================================================

# Step 1 — Reload raw data fresh (clean slate)
df = pd.read_csv("D:\Dataset-Mental-Disorders.csv")

# Step 2 — Strip hidden spaces from ALL text columns
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()

# Step 3 — Drop Patient Number (just a serial ID, useless for ML)
df = df.drop(columns=['Patient Number'])

print("✅ Step 1-3 done: Loaded, stripped, dropped ID column")
print(f"Shape: {df.shape}")

# ── Step 4 — Print EVERY column's unique values ──────────────
# We do this BEFORE any conversion so we see the truth
print("\n" + "="*55)
print("RAW UNIQUE VALUES IN EVERY COLUMN")
print("="*55)
for col in df.columns:
    print(f"\n  {col}:")
    print(f"     {df[col].unique()}")

✅ Step 1-3 done: Loaded, stripped, dropped ID column
Shape: (120, 18)

RAW UNIQUE VALUES IN EVERY COLUMN

  Sadness:
     ['Usually' 'Sometimes' 'Seldom' 'Most-Often']

  Euphoric:
     ['Seldom' 'Most-Often' 'Usually' 'Sometimes']

  Exhausted:
     ['Sometimes' 'Usually' 'Seldom' 'Most-Often']

  Sleep dissorder:
     ['Sometimes' 'Most-Often' 'Usually' 'Seldom']

  Mood Swing:
     ['YES' 'NO']

  Suicidal thoughts:
     ['YES' 'NO']

  Anorxia:
     ['NO' 'YES']

  Authority Respect:
     ['NO' 'YES']

  Try-Explanation:
     ['YES' 'NO']

  Aggressive Response:
     ['NO' 'YES']

  Ignore & Move-On:
     ['NO' 'YES']

  Nervous Break-down:
     ['YES' 'NO']

  Admit Mistakes:
     ['YES' 'NO']

  Overthinking:
     ['YES' 'NO']

  Sexual Activity:
     ['3 From 10' '4 From 10' '6 From 10' '5 From 10' '7 From 10' '8 From 10'
 '9 From 10' '2 From 10' '1 From 10']

  Concentration:
     ['3 From 10' '2 From 10' '5 From 10' '4 From 10' '7 From 10' '6 From 10'
 '1 From 10' '8 From 10']

In [30]:
# ============================================================
# COMPLETE CORRECT CLEANING — based on confirmed raw values
# ============================================================

# ── GROUP 1: Seldom / Sometimes / Usually / Most-Often ───────
frequency_map = {
    'Seldom':     0,   # least frequent
    'Sometimes':  1,
    'Usually':    2,
    'Most-Often': 3    # most frequent
}

frequency_cols = [
    'Sadness', 'Euphoric', 'Exhausted', 'Sleep dissorder',
    'Nervous Break-down', 'Try-Explanation'
]

for col in frequency_cols:
    df[col] = df[col].map(frequency_map)

print("✅ Group 1 done — frequency columns")


# ── GROUP 2: YES / NO ─────────────────────────────────────────
yes_no_map = {
    'YES': 1,
    'NO':  0
}

yes_no_cols = [
    'Mood Swing', 'Suicidal thoughts', 'Anorxia',
    'Authority Respect', 'Aggressive Response', 'Ignore & Move-On'
]

for col in yes_no_cols:
    df[col] = df[col].map(yes_no_map)

print("✅ Group 2 done — YES/NO columns")


# ── GROUP 3: "X From 10" → extract just the number ───────────
from_10_cols = [
    'Admit Mistakes', 'Overthinking', 'Sexual Activity',
    'Concentration', 'Optimisim'
]

for col in from_10_cols:
    # "4 From 10" → split by space → ["4","From","10"] → take "4" → int 4
    df[col] = df[col].str.split(' ').str[0].astype(int)

print("✅ Group 3 done — X From 10 columns")


# ── TARGET: Expert Diagnose → LabelEncoder ───────────────────
le = LabelEncoder()
df['Expert Diagnose'] = le.fit_transform(df['Expert Diagnose'])

print("✅ Target column encoded")
print("\nLabel mapping (memorize this!):")
for number, name in enumerate(le.classes_):
    print(f"   {number} → {name}")


# ── FINAL VERIFICATION ────────────────────────────────────────
print("\n" + "="*50)
print("VERIFICATION")
print("="*50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values: {df.isnull().sum().sum()}")  # MUST be 0
print("\nData types (all must be int64):")
print(df.dtypes)
print("\nFirst 3 rows:")
print(df.head(3))

✅ Group 1 done — frequency columns
✅ Group 2 done — YES/NO columns


ValueError: invalid literal for int() with base 10: 'YES'

In [31]:
# Check exact values of ONLY the Group 3 columns
# This will tell us immediately which one has YES/NO

group3_cols = [
    'Admit Mistakes', 'Overthinking', 
    'Sexual Activity', 'Concentration', 'Optimisim'
]

df_check = pd.read_csv("D:\Dataset-Mental-Disorders.csv")
for col in df_check.columns:
    if df_check[col].dtype == 'object':
        df_check[col] = df_check[col].str.strip()

for col in group3_cols:
    print(f"{col}: {df_check[col].unique()}")

Admit Mistakes: ['YES' 'NO']
Overthinking: ['YES' 'NO']
Sexual Activity: ['3 From 10' '4 From 10' '6 From 10' '5 From 10' '7 From 10' '8 From 10'
 '9 From 10' '2 From 10' '1 From 10']
Concentration: ['3 From 10' '2 From 10' '5 From 10' '4 From 10' '7 From 10' '6 From 10'
 '1 From 10' '8 From 10']
Optimisim: ['4 From 10' '5 From 10' '7 From 10' '2 From 10' '6 From 10' '9 From 10'
 '3 From 10' '8 From 10' '1 From 10']


In [32]:
# ============================================================
# COMPLETE CORRECT CLEANING — 100% verified groupings
# ============================================================

# Reload fresh one last time"D:\Dataset-Mental-Disorders.csv
df = pd.read_csv("D:\Dataset-Mental-Disorders.csv")

# Strip spaces from all text columns
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()

# Drop Patient Number
df = df.drop(columns=['Patient Number'])

print("✅ Fresh data loaded")

# ── GROUP 1: Seldom / Sometimes / Usually / Most-Often ───────
# These 6 columns use a frequency scale
frequency_map = {
    'Seldom':     0,
    'Sometimes':  1,
    'Usually':    2,
    'Most-Often': 3
}

frequency_cols = [
    'Sadness', 'Euphoric', 'Exhausted', 'Sleep dissorder',
    'Nervous Break-down', 'Try-Explanation'
]

for col in frequency_cols:
    df[col] = df[col].map(frequency_map)

print("✅ Group 1 done — 6 frequency columns")


# ── GROUP 2: YES / NO ─────────────────────────────────────────
# These 8 columns all use YES or NO
yes_no_map = {
    'YES': 1,
    'NO':  0
}

yes_no_cols = [
    'Mood Swing', 'Suicidal thoughts', 'Anorxia',
    'Authority Respect', 'Aggressive Response', 'Ignore & Move-On',
    'Admit Mistakes', 'Overthinking'      # ← confirmed YES/NO
]

for col in yes_no_cols:
    df[col] = df[col].map(yes_no_map)

print("✅ Group 2 done — 8 YES/NO columns")


# ── GROUP 3: "X From 10" → extract just the number ───────────
# Only these 3 columns use the X From 10 rating scale
from_10_cols = [
    'Sexual Activity',    # confirmed X From 10
    'Concentration',      # confirmed X From 10
    'Optimisim'           # confirmed X From 10
]

for col in from_10_cols:
    df[col] = df[col].str.split(' ').str[0].astype(int)

print("✅ Group 3 done — 3 rating columns")


# ── TARGET: Expert Diagnose ───────────────────────────────────
le = LabelEncoder()
df['Expert Diagnose'] = le.fit_transform(df['Expert Diagnose'])

print("✅ Target column encoded")
print("\nLabel mapping (save this — we need it later!):")
for number, name in enumerate(le.classes_):
    print(f"   {number} → {name}")


# ── FINAL VERIFICATION ────────────────────────────────────────
print("\n" + "="*50)
print("VERIFICATION")
print("="*50)

print(f"\nShape: {df.shape}")

missing = df.isnull().sum().sum()
print(f"\nMissing values: {missing}")
if missing == 0:
    print("   ✅ Perfect — no missing values!")
else:
    print("   ❌ Still have missing values — something is wrong")

print("\nData types:")
print(df.dtypes)

print("\nFirst 3 rows of fully cleaned data:")
print(df.head(3))

✅ Fresh data loaded
✅ Group 1 done — 6 frequency columns
✅ Group 2 done — 8 YES/NO columns
✅ Group 3 done — 3 rating columns
✅ Target column encoded

Label mapping (save this — we need it later!):
   0 → Bipolar Type-1
   1 → Bipolar Type-2
   2 → Depression
   3 → Normal

VERIFICATION

Shape: (120, 18)

Missing values: 240
   ❌ Still have missing values — something is wrong

Data types:
Sadness                  int64
Euphoric                 int64
Exhausted                int64
Sleep dissorder          int64
Mood Swing               int64
Suicidal thoughts        int64
Anorxia                  int64
Authority Respect        int64
Try-Explanation        float64
Aggressive Response      int64
Ignore & Move-On         int64
Nervous Break-down     float64
Admit Mistakes           int64
Overthinking             int64
Sexual Activity          int64
Concentration            int64
Optimisim                int64
Expert Diagnose          int64
dtype: object

First 3 rows of fully cleaned data:


In [33]:
# Find EXACTLY which columns still have missing values
print("Missing values per column:")
print(df.isnull().sum())

# Also check raw values of the two columns we're unsure about
df_check = pd.read_csv("D:\Dataset-Mental-Disorders.csv")
for col in df_check.columns:
    if df_check[col].dtype == 'object':
        df_check[col] = df_check[col].str.strip()

print("\nNervous Break-down raw values:")
print(df_check['Nervous Break-down'].unique())

print("\nTry-Explanation raw values:")
print(df_check['Try-Explanation'].unique())

Missing values per column:
Sadness                  0
Euphoric                 0
Exhausted                0
Sleep dissorder          0
Mood Swing               0
Suicidal thoughts        0
Anorxia                  0
Authority Respect        0
Try-Explanation        120
Aggressive Response      0
Ignore & Move-On         0
Nervous Break-down     120
Admit Mistakes           0
Overthinking             0
Sexual Activity          0
Concentration            0
Optimisim                0
Expert Diagnose          0
dtype: int64

Nervous Break-down raw values:
['YES' 'NO']

Try-Explanation raw values:
['YES' 'NO']


In [34]:
# ============================================================
# 100% VERIFIED FINAL CLEANING — all groups confirmed
# ============================================================

# Reload fresh
df = pd.read_csv("D:\Dataset-Mental-Disorders.csv")

# Strip spaces
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()

# Drop Patient Number
df = df.drop(columns=['Patient Number'])


# ── GROUP 1: Frequency scale — ONLY 4 columns ────────────────
frequency_map = {
    'Seldom':     0,
    'Sometimes':  1,
    'Usually':    2,
    'Most-Often': 3
}

frequency_cols = [
    'Sadness', 'Euphoric', 
    'Exhausted', 'Sleep dissorder'
]

for col in frequency_cols:
    df[col] = df[col].map(frequency_map)

print("✅ Group 1 done — 4 frequency columns")


# ── GROUP 2: YES/NO — 10 columns total ───────────────────────
yes_no_map = {'YES': 1, 'NO': 0}

yes_no_cols = [
    'Mood Swing',        # confirmed YES/NO
    'Suicidal thoughts', # confirmed YES/NO
    'Anorxia',           # confirmed YES/NO
    'Authority Respect', # confirmed YES/NO
    'Aggressive Response',# confirmed YES/NO
    'Ignore & Move-On',  # confirmed YES/NO
    'Admit Mistakes',    # confirmed YES/NO
    'Overthinking',      # confirmed YES/NO
    'Nervous Break-down',# confirmed YES/NO ← was wrong before
    'Try-Explanation'    # confirmed YES/NO ← was wrong before
]

for col in yes_no_cols:
    df[col] = df[col].map(yes_no_map)

print("✅ Group 2 done — 10 YES/NO columns")


# ── GROUP 3: X From 10 — 3 columns ───────────────────────────
from_10_cols = [
    'Sexual Activity',
    'Concentration',
    'Optimisim'
]

for col in from_10_cols:
    df[col] = df[col].str.split(' ').str[0].astype(int)

print("✅ Group 3 done — 3 rating columns")


# ── TARGET ────────────────────────────────────────────────────
le = LabelEncoder()
df['Expert Diagnose'] = le.fit_transform(df['Expert Diagnose'])

print("✅ Target column encoded")
print("\nLabel mapping:")
for number, name in enumerate(le.classes_):
    print(f"   {number} → {name}")


# ── FINAL VERIFICATION ────────────────────────────────────────
print("\n" + "="*50)
print("VERIFICATION")
print("="*50)

missing = df.isnull().sum().sum()
print(f"\nMissing values: {missing}")
if missing == 0:
    print("   ✅ PERFECT — zero missing values!")
else:
    print("   ❌ still have issues")
    print(df.isnull().sum())

print(f"\nShape: {df.shape}")
print("\nData types (all must be int64):")
print(df.dtypes)

print("\nFirst 3 rows:")
print(df.head(3))

✅ Group 1 done — 4 frequency columns
✅ Group 2 done — 10 YES/NO columns
✅ Group 3 done — 3 rating columns
✅ Target column encoded

Label mapping:
   0 → Bipolar Type-1
   1 → Bipolar Type-2
   2 → Depression
   3 → Normal

VERIFICATION

Missing values: 0
   ✅ PERFECT — zero missing values!

Shape: (120, 18)

Data types (all must be int64):
Sadness                int64
Euphoric               int64
Exhausted              int64
Sleep dissorder        int64
Mood Swing             int64
Suicidal thoughts      int64
Anorxia                int64
Authority Respect      int64
Try-Explanation        int64
Aggressive Response    int64
Ignore & Move-On       int64
Nervous Break-down     int64
Admit Mistakes         int64
Overthinking           int64
Sexual Activity        int64
Concentration          int64
Optimisim              int64
Expert Diagnose        int64
dtype: object

First 3 rows:
   Sadness  Euphoric  Exhausted  Sleep dissorder  Mood Swing  \
0        2         0          1            

## exploaratory data analysis

In [35]:
# ── How many patients per condition? ─────────────────────────
# This is the first chart every data scientist draws
# We want to confirm our classes are balanced

plt.figure(figsize=(8, 5))

# countplot counts how many rows belong to each class
# and draws a bar for each one
ax = sns.countplot(
    x='Expert Diagnose', 
    data=df,
    palette=['#4C72B0', '#DD8452', '#55A868', '#C44E52']
)

# Replace 0,1,2,3 on x-axis with actual condition names
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['Bipolar Type-1', 'Bipolar Type-2', 
                     'Depression', 'Normal'], fontsize=11)

# Add count number on top of each bar
for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height())}',        # the number to show
        (p.get_x() + p.get_width() / 2, # x position (center of bar)
         p.get_height() + 0.3),          # y position (just above bar)
        ha='center', fontsize=12, fontweight='bold'
    )

plt.title('Number of patients per mental health condition', 
          fontsize=14, fontweight='bold')
plt.xlabel('Condition', fontsize=12)
plt.ylabel('Number of patients', fontsize=12)
plt.tight_layout()
plt.show()

print("\nExact counts:")
counts = df['Expert Diagnose'].value_counts().sort_index()
for idx, count in counts.items():
    print(f" {le.classes_[idx]}: {count} patients")

C:\Users\Mohit\AppData\Local\Temp\ipykernel_19772\775234999.py:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(



Exact counts:
 Bipolar Type-1: 28 patients
 Bipolar Type-2: 31 patients
 Depression: 31 patients
 Normal: 30 patients


C:\Users\Mohit\AppData\Local\Temp\ipykernel_19772\775234999.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [36]:
# This chart shows the AVERAGE value of each symptom
# grouped by condition
# It's the most important chart for understanding
# WHICH symptoms separate one condition from another

# Select only the feature columns (not the target)
feature_cols = [col for col in df.columns if col != 'Expert Diagnose']

# Calculate average value of each feature per condition
# groupby() groups rows by condition, then mean() averages each column
avg_by_condition = df.groupby('Expert Diagnose')[feature_cols].mean()

# Replace 0,1,2,3 index with actual condition names
avg_by_condition.index = le.classes_

plt.figure(figsize=(14, 6))

# heatmap shows a grid where:
# - rows = conditions
# - columns = symptoms  
# - color = average value (darker = higher)
sns.heatmap(
    avg_by_condition,
    annot=True,          # show numbers inside each cell
    fmt='.1f',           # round to 1 decimal place
    cmap='YlOrRd',       # color: yellow (low) → orange → red (high)
    linewidths=0.5,      # thin lines between cells
    cbar_kws={'label': 'Average value'}
)

plt.title('Average symptom intensity per condition', 
          fontsize=14, fontweight='bold')
plt.xlabel('Symptom', fontsize=12)
plt.ylabel('Condition', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout()
plt.show()

C:\Users\Mohit\AppData\Local\Temp\ipykernel_19772\1245618705.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [37]:
# Check which columns are still object type
print("Columns that are still text (object):")
object_cols = df.select_dtypes(include='object').columns.tolist()

if len(object_cols) == 0:
    print("   ✅ None — all columns are numbers!")
else:
    print(f"   ❌ These columns are still text: {object_cols}")
    for col in object_cols:
        print(f"\n   {col}: {df[col].unique()}")

Columns that are still text (object):
   ✅ None — all columns are numbers!


In [38]:
# ── Fix Admit Mistakes and Overthinking (YES/NO) ─────────────
yes_no_map = {'YES': 1, 'NO': 0}

for col in ['Admit Mistakes', 'Overthinking']:
    df[col] = df[col].map(yes_no_map)

print("✅ Admit Mistakes and Overthinking fixed")


# ── Fix Sexual Activity, Concentration, Optimisim (X From 10)─
for col in ['Sexual Activity', 'Concentration', 'Optimisim']:
    df[col] = df[col].str.split(' ').str[0].astype(int)

print("✅ Sexual Activity, Concentration, Optimisim fixed")


# ── Fix Expert Diagnose (target label) ───────────────────────
le = LabelEncoder()
df['Expert Diagnose'] = le.fit_transform(df['Expert Diagnose'])

print("✅ Expert Diagnose encoded")
print("\nLabel mapping:")
for number, name in enumerate(le.classes_):
    print(f"   {number} → {name}")


# ── Verify everything is fixed ────────────────────────────────
print("\n" + "="*45)
remaining_objects = df.select_dtypes(include='object').columns.tolist()

if len(remaining_objects) == 0:
    print("✅ ALL COLUMNS ARE NOW NUMBERS — data is 100% clean!")
else:
    print(f"❌ Still text: {remaining_objects}")

print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Shape: {df.shape}")

✅ Admit Mistakes and Overthinking fixed


AttributeError: Can only use .str accessor with string values!

In [39]:
# ============================================================
# MASTER CELL — Run this ONE cell after Kernel Restart
# This does everything: import, load, clean, verify
# ============================================================

# ── IMPORTS ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix)
np.random.seed(42)
print("✅ Libraries imported")

# ── LOAD ─────────────────────────────────────────────────────
df = pd.read_csv("D:\Dataset-Mental-Disorders.csv")
print(f"✅ Data loaded — shape: {df.shape}")

# ── STRIP SPACES ─────────────────────────────────────────────
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()
print("✅ Spaces stripped")

# ── DROP PATIENT NUMBER ───────────────────────────────────────
df = df.drop(columns=['Patient Number'])
print("✅ Patient Number dropped")

# ── GROUP 1: Frequency columns (4 columns) ───────────────────
# Values: Seldom / Sometimes / Usually / Most-Often → 0/1/2/3
frequency_map = {
    'Seldom':     0,
    'Sometimes':  1,
    'Usually':    2,
    'Most-Often': 3
}
frequency_cols = [
    'Sadness', 'Euphoric',
    'Exhausted', 'Sleep dissorder'
]
for col in frequency_cols:
    df[col] = df[col].map(frequency_map)
print("✅ Group 1 done — 4 frequency columns → 0/1/2/3")

# ── GROUP 2: YES/NO columns (10 columns) ─────────────────────
# Values: YES / NO → 1/0
yes_no_map = {'YES': 1, 'NO': 0}
yes_no_cols = [
    'Mood Swing', 'Suicidal thoughts',
    'Anorxia', 'Authority Respect',
    'Aggressive Response', 'Ignore & Move-On',
    'Admit Mistakes', 'Overthinking',
    'Nervous Break-down', 'Try-Explanation'
]
for col in yes_no_cols:
    df[col] = df[col].map(yes_no_map)
print("✅ Group 2 done — 10 YES/NO columns → 0/1")

# ── GROUP 3: Rating columns (3 columns) ──────────────────────
# Values: "X From 10" → X (just the number)
from_10_cols = [
    'Sexual Activity',
    'Concentration',
    'Optimisim'
]
for col in from_10_cols:
    df[col] = df[col].str.split(' ').str[0].astype(int)
print("✅ Group 3 done — 3 rating columns → 1 to 9")

# ── TARGET: Expert Diagnose ───────────────────────────────────
le = LabelEncoder()
df['Expert Diagnose'] = le.fit_transform(df['Expert Diagnose'])
print("✅ Target encoded")
print("\nLabel mapping (save this!):")
for number, name in enumerate(le.classes_):
    print(f"   {number} → {name}")

# ── FINAL VERIFICATION ────────────────────────────────────────
print("\n" + "="*50)
print("FINAL VERIFICATION")
print("="*50)

# Check 1 — missing values
missing = df.isnull().sum().sum()
print(f"\n Missing values : {missing}",
      "✅" if missing == 0 else "❌")

# Check 2 — shape
print(f" Shape          : {df.shape}",
      "✅" if df.shape == (120, 17) else "❌")

# Check 3 — all int64
all_int = all(df.dtypes == 'int64')
print(f" All int64      : {all_int}",
      "✅" if all_int else "❌")

# Check 4 — show first 3 rows
print("\nFirst 3 rows:")
print(df.head(3))

# Check 5 — show any remaining issues
if missing > 0:
    print("\nColumns with missing values:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

✅ Libraries imported
✅ Data loaded — shape: (120, 19)
✅ Spaces stripped
✅ Patient Number dropped
✅ Group 1 done — 4 frequency columns → 0/1/2/3
✅ Group 2 done — 10 YES/NO columns → 0/1
✅ Group 3 done — 3 rating columns → 1 to 9
✅ Target encoded

Label mapping (save this!):
   0 → Bipolar Type-1
   1 → Bipolar Type-2
   2 → Depression
   3 → Normal

FINAL VERIFICATION

 Missing values : 0 ✅
 Shape          : (120, 18) ❌
 All int64      : True ✅

First 3 rows:
   Sadness  Euphoric  Exhausted  Sleep dissorder  Mood Swing  \
0        2         0          1                1           1   
1        2         0          2                1           0   
2        1         3          1                1           1   

   Suicidal thoughts  Anorxia  Authority Respect  Try-Explanation  \
0                  1        0                  0                1   
1                  1        0                  0                0   
2                  0        0                  0                1   

   

In [40]:
# eda cell 1 - class distribution
# How many patients per condition?
plt.figure(figsize=(8, 5))

ax = sns.countplot(
    x='Expert Diagnose',
    data=df,
    palette=['#4C72B0', '#DD8452', '#55A868', '#C44E52']
)

# Replace numbers with real condition names on x-axis
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(le.classes_, fontsize=11)

# Show count number on top of each bar
for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height())}',
        (p.get_x() + p.get_width() / 2, p.get_height() + 0.3),
        ha='center', fontsize=12, fontweight='bold'
    )

plt.title('Number of patients per condition', fontsize=14, fontweight='bold')
plt.xlabel('Condition', fontsize=12)
plt.ylabel('Number of patients', fontsize=12)
plt.tight_layout()
plt.show()

C:\Users\Mohit\AppData\Local\Temp\ipykernel_19772\2129278218.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(
C:\Users\Mohit\AppData\Local\Temp\ipykernel_19772\2129278218.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [41]:
# eda cell 2-  symptom heatmap
# Average value of each symptom grouped by condition
# This is the most important chart — shows which symptoms
# separate one condition from another

feature_cols = [col for col in df.columns if col != 'Expert Diagnose']

# Group by condition and calculate mean of each symptom
avg_by_condition = df.groupby('Expert Diagnose')[feature_cols].mean()

# Replace 0,1,2,3 with real condition names
avg_by_condition.index = le.classes_

plt.figure(figsize=(16, 5))

sns.heatmap(
    avg_by_condition,
    annot=True,        # show numbers in each cell
    fmt='.1f',         # 1 decimal place
    cmap='YlOrRd',     # yellow=low, red=high
    linewidths=0.5,
    cbar_kws={'label': 'Average value'}
)

plt.title('Average symptom intensity per condition',
          fontsize=14, fontweight='bold')
plt.xlabel('Symptom', fontsize=11)
plt.ylabel('Condition', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout()
plt.show()

C:\Users\Mohit\AppData\Local\Temp\ipykernel_19772\3523605909.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [42]:
# eda cell 3 - top symptom comparosion
# Bar chart comparing 4 key symptoms across all conditions
# We pick the most clinically interesting ones

key_symptoms = ['Sadness', 'Euphoric', 'Suicidal thoughts', 'Mood Swing']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for i, symptom in enumerate(key_symptoms):
    # Average value of this symptom per condition
    avg = df.groupby('Expert Diagnose')[symptom].mean()
    
    axes[i].bar(
        range(len(le.classes_)),
        avg.values,
        color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'],
        edgecolor='white',
        linewidth=1.2
    )
    
    # Add value labels on top of each bar
    for j, val in enumerate(avg.values):
        axes[i].text(
            j, val + 0.02,
            f'{val:.2f}',
            ha='center', fontsize=10, fontweight='bold'
        )
    
    axes[i].set_title(symptom, fontsize=12, fontweight='bold')
    axes[i].set_xticks(range(len(le.classes_)))
    axes[i].set_xticklabels(le.classes_, rotation=30,
                             ha='right', fontsize=9)
    axes[i].set_ylabel('Average value', fontsize=10)

plt.suptitle('Key symptom comparison across conditions',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

C:\Users\Mohit\AppData\Local\Temp\ipykernel_19772\2370407368.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [43]:
#eda cell 4 - correlaiton heatmap
# Correlation tells us how strongly two columns move together
# +1 = when one goes up, the other goes up too
# -1 = when one goes up, the other goes down
#  0 = no relationship at all

plt.figure(figsize=(14, 10))

# .corr() calculates correlation between every pair of columns
correlation_matrix = df.corr()

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',   # blue=negative, white=zero, red=positive
    center=0,          # white is at 0
    linewidths=0.5,
    annot_kws={'size': 8}
)

plt.title('Correlation between all features and target',
          fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

# Print top features most correlated with Expert Diagnose
print("Top features most correlated with Expert Diagnose:")
correlations = df.corr()['Expert Diagnose'].drop('Expert Diagnose')
correlations_sorted = correlations.abs().sort_values(ascending=False)
for feat, val in correlations_sorted.head(8).items():
    print(f"   {feat}: {val:.3f}")

Top features most correlated with Expert Diagnose:
   Mood Swing: 0.801
   Aggressive Response: 0.355
   Concentration: 0.322
   Nervous Break-down: 0.281
   Sleep dissorder: 0.255
   Ignore & Move-On: 0.241
   Suicidal thoughts: 0.241
   Authority Respect: 0.237


C:\Users\Mohit\AppData\Local\Temp\ipykernel_19772\4251062015.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Building ml models

In [44]:
# Cell — Split data into train and test sets
# ── Separate features (X) from target (y) ────────────────────
# X = all columns EXCEPT Expert Diagnose (these are our inputs)
# y = only Expert Diagnose column (this is what we want to predict)

X = df.drop(columns=['Expert Diagnose'])
y = df['Expert Diagnose']

print(f"X shape (features): {X.shape}")
# Should be (120, 17) — 120 patients, 17 features

print(f"y shape (target):   {y.shape}")
# Should be (120,) — 120 labels

# ── Split into training and testing sets ─────────────────────
# We give 80% of data to train the model
# We HIDE the remaining 20% to test how well it learned
# test_size=0.2 means 20% goes to testing
# random_state=42 means the split is the same every time you run it
# stratify=y means each condition gets equal representation
# in both train and test sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTraining set   : {X_train.shape[0]} patients (80%)")
print(f"Testing set    : {X_test.shape[0]} patients (20%)")
print(f"\nCondition distribution in training set:")
for i, name in enumerate(le.classes_):
    count = (y_train == i).sum()
    print(f"   {name}: {count} patients")
print(f"\nCondition distribution in testing set:")
for i, name in enumerate(le.classes_):
    count = (y_test == i).sum()
    print(f"   {name}: {count} patients")

X shape (features): (120, 17)
y shape (target):   (120,)

Training set   : 96 patients (80%)
Testing set    : 24 patients (20%)

Condition distribution in training set:
   Bipolar Type-1: 22 patients
   Bipolar Type-2: 25 patients
   Depression: 25 patients
   Normal: 24 patients

Condition distribution in testing set:
   Bipolar Type-1: 6 patients
   Bipolar Type-2: 6 patients
   Depression: 6 patients
   Normal: 6 patients


## Cell — Train Model 1: Logistic Regression

In [45]:
# ── MODEL 1: Logistic Regression ─────────────────────────────
# This is the simplest ML model — a great baseline
# Think of it as drawing straight lines to separate conditions
# If our simple model does well, it means the data is well structured

print("Training Model 1: Logistic Regression...")

model_lr = LogisticRegression(
    max_iter=1000,    # how many times it adjusts itself while learning
    random_state=42
)

# .fit() is where the actual LEARNING happens
# We show the model the training features AND the correct answers
# It finds patterns that connect features to conditions
model_lr.fit(X_train, y_train)

# .predict() uses what the model learned to guess on NEW data
# The model has NEVER seen X_test before — this is a real test
y_pred_lr = model_lr.predict(X_test)

# accuracy_score compares predictions vs real answers
acc_lr = accuracy_score(y_test, y_pred_lr)

print(f"✅ Logistic Regression trained!")
print(f"   Accuracy: {acc_lr*100:.1f}%")

Training Model 1: Logistic Regression...
✅ Logistic Regression trained!
   Accuracy: 91.7%


## Cell — Train Model 2: Random Forest

In [46]:
# ── MODEL 2: Random Forest ────────────────────────────────────
# Random Forest builds MANY decision trees and combines their votes
# Think of it like asking 100 doctors and going with the majority opinion
# Much more powerful than a single Logistic Regression

print("Training Model 2: Random Forest...")

model_rf = RandomForestClassifier(
    n_estimators=100,   # build 100 decision trees
    random_state=42
)

model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"✅ Random Forest trained!")
print(f"   Accuracy: {acc_rf*100:.1f}%")

Training Model 2: Random Forest...
✅ Random Forest trained!
   Accuracy: 87.5%


## Cell — Train Model 3: Gradient Boosting

In [47]:
# ── MODEL 3: Gradient Boosting ────────────────────────────────
# This is the most powerful of the three models
# It builds trees ONE BY ONE — each new tree focuses on fixing
# the mistakes made by the previous tree
# Like a student who only studies topics they got wrong last time

print("Training Model 3: Gradient Boosting...")

model_gb = GradientBoostingClassifier(
    n_estimators=100,   # build 100 trees sequentially
    learning_rate=0.1,  # how much each tree corrects the previous
    random_state=42
)

model_gb.fit(X_train, y_train)
y_pred_gb = model_gb.predict(X_test)
acc_gb = accuracy_score(y_test, y_pred_gb)

print(f"✅ Gradient Boosting trained!")
print(f"   Accuracy: {acc_gb*100:.1f}%")

Training Model 3: Gradient Boosting...
✅ Gradient Boosting trained!
   Accuracy: 91.7%


## Cell — Compare all 3 models

In [48]:
# ── Compare all 3 models side by side ────────────────────────
print("=" * 45)
print("MODEL COMPARISON")
print("=" * 45)

models = {
    'Logistic Regression': acc_lr,
    'Random Forest':       acc_rf,
    'Gradient Boosting':   acc_gb
}

best_model_name = max(models, key=models.get)

for name, acc in models.items():
    marker = "⭐ BEST" if name == best_model_name else ""
    print(f"  {name:25s}: {acc*100:.1f}%  {marker}")

print("=" * 45)

# Bar chart of model accuracies
plt.figure(figsize=(8, 5))
bars = plt.bar(
    models.keys(),
    [v * 100 for v in models.values()],
    color=['#4C72B0', '#55A868', '#DD8452'],
    edgecolor='white',
    linewidth=1.5,
    width=0.5
)

# Add accuracy % on top of each bar
for bar, val in zip(bars, models.values()):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val*100:.1f}%',
        ha='center', fontsize=13, fontweight='bold'
    )

plt.title('Model accuracy comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)', fontsize=12)
plt.ylim(0, 115)
plt.xticks(fontsize=11)
plt.tight_layout()
plt.show()

MODEL COMPARISON
  Logistic Regression      : 91.7%  ⭐ BEST
  Random Forest            : 87.5%  
  Gradient Boosting        : 91.7%  


C:\Users\Mohit\AppData\Local\Temp\ipykernel_19772\1867319245.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 📊 Cell — Detailed evaluation of the best model

In [49]:
# We'll evaluate Logistic Regression in detail
# since it's our best model (tied but simpler = winner)

print("="*55)
print("DETAILED EVALUATION — Logistic Regression")
print("="*55)

# Classification report gives us 4 metrics for EACH condition:
# Precision, Recall, F1-score, Support
print("\nClassification Report:")
print(classification_report(
    y_test,           # real answers
    y_pred_lr,        # model's predictions
    target_names=le.classes_  # use real condition names
))

DETAILED EVALUATION — Logistic Regression

Classification Report:
                precision    recall  f1-score   support

Bipolar Type-1       1.00      1.00      1.00         6
Bipolar Type-2       1.00      1.00      1.00         6
    Depression       0.75      1.00      0.86         6
        Normal       1.00      0.67      0.80         6

      accuracy                           0.92        24
     macro avg       0.94      0.92      0.91        24
  weighted avg       0.94      0.92      0.91        24



## 📊 Cell — Confusion Matrix

In [50]:
# Confusion matrix shows exactly WHERE the model gets confused
# Which conditions does it mix up with each other?

cm = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,           # show numbers in each cell
    fmt='d',              # show as integers not decimals
    cmap='Blues',         # white=0, dark blue=high
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    linewidths=1,
    linecolor='white',
    cbar=False
)

plt.title('Confusion Matrix — Logistic Regression',
          fontsize=14, fontweight='bold')
plt.ylabel('Actual condition', fontsize=12)
plt.xlabel('Predicted condition', fontsize=12)
plt.xticks(rotation=30, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

# Plain English explanation of the matrix
print("\nHow to read this matrix:")
print("• Diagonal cells (top-left to bottom-right) = CORRECT predictions")
print("• Off-diagonal cells = MISTAKES (what it predicted vs what was real)")
print()
for i, actual in enumerate(le.classes_):
    for j, predicted in enumerate(le.classes_):
        if i != j and cm[i][j] > 0:
            print(f"  ❌ {cm[i][j]} patient(s) with {actual} "
                  f"were wrongly predicted as {predicted}")


How to read this matrix:
• Diagonal cells (top-left to bottom-right) = CORRECT predictions
• Off-diagonal cells = MISTAKES (what it predicted vs what was real)

  ❌ 2 patient(s) with Normal were wrongly predicted as Depression


C:\Users\Mohit\AppData\Local\Temp\ipykernel_19772\3299934176.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 📊 Cell — Feature importance (what the model relies on most)

In [51]:
# Random Forest gives us "feature importance" scores
# This tells us which features the model used most
# to make its decisions

importances = model_rf.feature_importances_
feature_names = X.columns

# Sort features by importance (highest first)
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))

bars = plt.bar(
    range(len(feature_names)),
    importances[indices],
    color=['#C44E52' if importances[i] > 0.1 else
           '#DD8452' if importances[i] > 0.05 else
           '#4C72B0' for i in indices],
    edgecolor='white',
    linewidth=1
)

plt.xticks(
    range(len(feature_names)),
    [feature_names[i] for i in indices],
    rotation=45, ha='right', fontsize=10
)

plt.title('Feature importance — Random Forest',
          fontsize=14, fontweight='bold')
plt.ylabel('Importance score', fontsize=12)
plt.xlabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

print("\nFeature importance ranking:")
for rank, i in enumerate(indices):
    bar = "█" * int(importances[i] * 100)
    print(f"  {rank+1:2d}. {feature_names[i]:25s} "
          f"{importances[i]:.4f}  {bar}")


Feature importance ranking:
   1. Mood Swing                0.2645  ██████████████████████████
   2. Optimisim                 0.1213  ████████████
   3. Sexual Activity           0.0894  ████████
   4. Sadness                   0.0777  ███████
   5. Suicidal thoughts         0.0737  ███████
   6. Euphoric                  0.0641  ██████
   7. Exhausted                 0.0475  ████
   8. Concentration             0.0470  ████
   9. Authority Respect         0.0384  ███
  10. Sleep dissorder           0.0371  ███
  11. Aggressive Response       0.0303  ███
  12. Nervous Break-down        0.0248  ██
  13. Ignore & Move-On          0.0212  ██
  14. Overthinking              0.0188  █
  15. Anorxia                   0.0160  █
  16. Try-Explanation           0.0153  █
  17. Admit Mistakes            0.0130  █


C:\Users\Mohit\AppData\Local\Temp\ipykernel_19772\3169990265.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 📊 Cell — Cross validation (the most trustworthy accuracy test)

In [52]:
# Cross validation tests the model more fairly
# Instead of one 80/20 split, it does 5 different splits
# and averages the accuracy across all 5
# This gives a much more reliable estimate of real performance

from sklearn.model_selection import cross_val_score

print("Running 5-fold cross validation on all models...\n")

# cv=5 means split data 5 ways and test 5 times
cv_lr = cross_val_score(model_lr, X, y, cv=5, scoring='accuracy')
cv_rf = cross_val_score(model_rf, X, y, cv=5, scoring='accuracy')
cv_gb = cross_val_score(model_gb, X, y, cv=5, scoring='accuracy')

print("Logistic Regression:")
print(f"   Scores : {[f'{s*100:.1f}%' for s in cv_lr]}")
print(f"   Average: {cv_lr.mean()*100:.1f}% "
      f"(+/- {cv_lr.std()*100:.1f}%)")

print("\nRandom Forest:")
print(f"   Scores : {[f'{s*100:.1f}%' for s in cv_rf]}")
print(f"   Average: {cv_rf.mean()*100:.1f}% "
      f"(+/- {cv_rf.std()*100:.1f}%)")

print("\nGradient Boosting:")
print(f"   Scores : {[f'{s*100:.1f}%' for s in cv_gb]}")
print(f"   Average: {cv_gb.mean()*100:.1f}% "
      f"(+/- {cv_gb.std()*100:.1f}%)")

print("\n" + "="*45)
print("FINAL VERDICT")
print("="*45)
print(f"  Logistic Regression: {cv_lr.mean()*100:.1f}%")
print(f"  Random Forest      : {cv_rf.mean()*100:.1f}%")
print(f"  Gradient Boosting  : {cv_gb.mean()*100:.1f}%")

best_cv = max([
    ('Logistic Regression', cv_lr.mean()),
    ('Random Forest',       cv_rf.mean()),
    ('Gradient Boosting',   cv_gb.mean())
], key=lambda x: x[1])

print(f"\n  ⭐ Best model overall: {best_cv[0]} "
      f"({best_cv[1]*100:.1f}%)")

Running 5-fold cross validation on all models...

Logistic Regression:
   Scores : ['79.2%', '100.0%', '83.3%', '100.0%', '87.5%']
   Average: 90.0% (+/- 8.6%)

Random Forest:
   Scores : ['91.7%', '91.7%', '79.2%', '91.7%', '87.5%']
   Average: 88.3% (+/- 4.9%)

Gradient Boosting:
   Scores : ['83.3%', '87.5%', '87.5%', '91.7%', '91.7%']
   Average: 88.3% (+/- 3.1%)

FINAL VERDICT
  Logistic Regression: 90.0%
  Random Forest      : 88.3%
  Gradient Boosting  : 88.3%

  ⭐ Best model overall: Logistic Regression (90.0%)


##  Save the model and make predictions

In [53]:
import pickle

# pickle saves our trained model to a file on your computer
# So we don't have to retrain it every time we want to use it
# Think of it like saving a Word document — the work is preserved

# Save the model
with open('mental_health_model.pkl', 'wb') as f:
    # 'wb' means "write binary" — ML models are saved as binary files
    pickle.dump(model_lr, f)

# Save the label encoder too — we need it to convert
# numbers back to condition names when predicting
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Save the feature column names — we need these to make sure
# new input data has columns in the exact right order
with open('feature_columns.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)

print("✅ Model saved as 'mental_health_model.pkl'")
print("✅ Label encoder saved as 'label_encoder.pkl'")
print("✅ Feature columns saved as 'feature_columns.pkl'")
print("\nThese 3 files are now in your project folder.")
print("You can load them anytime without retraining!")

✅ Model saved as 'mental_health_model.pkl'
✅ Label encoder saved as 'label_encoder.pkl'
✅ Feature columns saved as 'feature_columns.pkl'

These 3 files are now in your project folder.
You can load them anytime without retraining!


## Cell — Build the prediction function

In [54]:
# This is the HEART of our project
# A function that takes a patient's answers and returns
# their predicted mental health condition

def predict_mental_health(patient_answers):
    """
    Takes a dictionary of patient answers and returns
    the predicted mental health condition.
    
    Parameters:
        patient_answers: dict with all 17 feature values
    
    Returns:
        condition name, confidence percentage
    """
    
    # Step 1 — Convert the answers dict into a DataFrame
    # The model expects a DataFrame, not a plain dictionary
    input_df = pd.DataFrame([patient_answers])
    
    # Step 2 — Make sure columns are in the exact same order
    # as when we trained the model
    input_df = input_df[list(X.columns)]
    
    # Step 3 — Get the prediction (returns a number 0/1/2/3)
    prediction_number = model_lr.predict(input_df)[0]
    
    # Step 4 — Convert number back to condition name
    condition_name = le.inverse_transform([prediction_number])[0]
    
    # Step 5 — Get confidence (probability for each condition)
    # predict_proba returns probability for each class
    probabilities = model_lr.predict_proba(input_df)[0]
    confidence = probabilities[prediction_number] * 100
    
    # Step 6 — Build a readable result
    print("=" * 50)
    print("MENTAL HEALTH ASSESSMENT RESULT")
    print("=" * 50)
    print(f"\n  Predicted condition : {condition_name}")
    print(f"  Model confidence    : {confidence:.1f}%")
    print("\n  Probability breakdown:")
    for i, cond in enumerate(le.classes_):
        bar = "█" * int(probabilities[i] * 20)
        print(f"    {cond:15s}: {probabilities[i]*100:5.1f}%  {bar}")
    print("=" * 50)
    
    return condition_name, confidence


print("✅ predict_mental_health() function is ready!")

✅ predict_mental_health() function is ready!


## Cell — Test with a depressed patient profile

In [55]:
# Let's test with a patient who shows strong depression symptoms
# Frequency:  Seldom=0  Sometimes=1  Usually=2  Most-Often=3
# YES/NO:     NO=0  YES=1
# Rating:     actual number 1-9

depressed_patient = {
    # Frequency symptoms (0-3)
    'Sadness':            3,   # Most-Often — very sad
    'Euphoric':           0,   # Seldom — rarely happy
    'Exhausted':          3,   # Most-Often — always tired
    'Sleep dissorder':    2,   # Usually — sleep problems

    # YES/NO symptoms
    'Mood Swing':         0,   # NO — no mood swings
    'Suicidal thoughts':  1,   # YES — has suicidal thoughts
    'Anorxia':            1,   # YES — eating disorder
    'Authority Respect':  1,   # YES
    'Aggressive Response':0,   # NO — not aggressive
    'Ignore & Move-On':   0,   # NO
    'Admit Mistakes':     1,   # YES
    'Overthinking':       1,   # YES — overthinks a lot
    'Nervous Break-down': 1,   # YES
    'Try-Explanation':    1,   # YES

    # Rating symptoms (1-9)
    'Sexual Activity':    2,   # low
    'Concentration':      2,   # poor concentration
    'Optimisim':          2    # very low optimism
}

result = predict_mental_health(depressed_patient)

MENTAL HEALTH ASSESSMENT RESULT

  Predicted condition : Bipolar Type-2
  Model confidence    : 54.2%

  Probability breakdown:
    Bipolar Type-1 :   0.0%  
    Bipolar Type-2 :  54.2%  ██████████
    Depression     :  45.8%  █████████
    Normal         :   0.1%  


## Cell — Test with a normal/healthy patient profile

In [56]:
# Now test with someone who seems mentally healthy

healthy_patient = {
    # Frequency symptoms (0-3)
    'Sadness':            0,   # Seldom — rarely sad
    'Euphoric':           1,   # Sometimes
    'Exhausted':          1,   # Sometimes
    'Sleep dissorder':    0,   # Seldom — sleeps well

    # YES/NO symptoms
    'Mood Swing':         0,   # NO
    'Suicidal thoughts':  0,   # NO
    'Anorxia':            0,   # NO
    'Authority Respect':  1,   # YES
    'Aggressive Response':0,   # NO
    'Ignore & Move-On':   1,   # YES
    'Admit Mistakes':     1,   # YES
    'Overthinking':       0,   # NO
    'Nervous Break-down': 0,   # NO
    'Try-Explanation':    1,   # YES

    # Rating symptoms (1-9)
    'Sexual Activity':    7,   # healthy
    'Concentration':      8,   # good focus
    'Optimisim':          8    # very optimistic
}

result = predict_mental_health(healthy_patient)

MENTAL HEALTH ASSESSMENT RESULT

  Predicted condition : Normal
  Model confidence    : 99.5%

  Probability breakdown:
    Bipolar Type-1 :   0.4%  
    Bipolar Type-2 :   0.0%  
    Depression     :   0.1%  
    Normal         :  99.5%  ███████████████████


## Cell — Test with a Bipolar patient profile

In [57]:
# Bipolar is characterized by extreme mood swings
# alternating between very high (euphoric) and very low periods

bipolar_patient = {
    # Frequency symptoms (0-3)
    'Sadness':            2,   # Usually
    'Euphoric':           3,   # Most-Often — key bipolar sign
    'Exhausted':          2,   # Usually
    'Sleep dissorder':    3,   # Most-Often — very disrupted sleep

    # YES/NO symptoms
    'Mood Swing':         1,   # YES — the strongest bipolar indicator
    'Suicidal thoughts':  1,   # YES
    'Anorxia':            0,   # NO
    'Authority Respect':  0,   # NO — defiant
    'Aggressive Response':1,   # YES — very aggressive
    'Ignore & Move-On':   0,   # NO
    'Admit Mistakes':     0,   # NO
    'Overthinking':       1,   # YES
    'Nervous Break-down': 1,   # YES
    'Try-Explanation':    0,   # NO

    # Rating symptoms (1-9)
    'Sexual Activity':    8,   # elevated — common in bipolar
    'Concentration':      3,   # poor
    'Optimisim':          6    # fluctuating
}

result = predict_mental_health(bipolar_patient)

MENTAL HEALTH ASSESSMENT RESULT

  Predicted condition : Bipolar Type-1
  Model confidence    : 99.0%

  Probability breakdown:
    Bipolar Type-1 :  99.0%  ███████████████████
    Bipolar Type-2 :   0.7%  
    Depression     :   0.2%  
    Normal         :   0.0%  


## FINAL CELL - INTERACTIVE MENTAL HEALTH QUESTIONNAIRE

In [59]:
# ============================================================
# INTERACTIVE MENTAL HEALTH QUESTIONNAIRE
# Run this cell and answer each question as it appears
# At the end you get your mental health assessment
# ============================================================

def run_questionnaire():
    """
    Interactive questionnaire that collects patient answers
    one question at a time and returns mental health prediction
    """
    
    print("="*55)
    print("   MENTAL HEALTH ASSESSMENT QUESTIONNAIRE")
    print("="*55)
    print("Answer each question honestly.")
    print("Your answers are used to assess your mental health.")
    print("="*55)
    
    answers = {}
    
    # ── SECTION 1: Frequency questions ───────────────────────
    print("\n📋 SECTION 1 — How often do you experience these?")
    print("   Options: 0=Seldom  1=Sometimes  2=Usually  3=Most-Often")
    print()
    
    frequency_questions = {
        'Sadness':          "1. How often do you feel sad or empty?",
        'Euphoric':         "2. How often do you feel unusually happy or high energy?",
        'Exhausted':        "3. How often do you feel physically or mentally exhausted?",
        'Sleep dissorder':  "4. How often do you have trouble sleeping?"
    }
    
    for col, question in frequency_questions.items():
        while True:
            # input() pauses and waits for the user to type something
            try:
                val = int(input(f"   {question}\n   Your answer (0/1/2/3): "))
                if val in [0, 1, 2, 3]:
                    answers[col] = val
                    break
                else:
                    print("   ⚠️  Please enter 0, 1, 2, or 3 only")
            except ValueError:
                print("   ⚠️  Please enter a number (0, 1, 2, or 3)")
    
    # ── SECTION 2: YES/NO questions ───────────────────────────
    print("\n📋 SECTION 2 — Yes or No questions")
    print("   Options: 0=No  1=Yes")
    print()
    
    yes_no_questions = {
        'Mood Swing':          "5.  Do you experience sudden extreme mood swings?",
        'Suicidal thoughts':   "6.  Have you had thoughts of self-harm or suicide?",
        'Anorxia':             "7.  Do you have unusual eating habits or loss of appetite?",
        'Authority Respect':   "8.  Do you generally respect authority figures?",
        'Aggressive Response': "9.  Do you respond aggressively when stressed or upset?",
        'Ignore & Move-On':    "10. Do you tend to ignore problems and move on?",
        'Admit Mistakes':      "11. Can you easily admit when you make mistakes?",
        'Overthinking':        "12. Do you overthink situations excessively?",
        'Nervous Break-down':  "13. Do you experience nervous breakdowns?",
        'Try-Explanation':     "14. Do you try to explain your behavior to others?"
    }
    
    for col, question in yes_no_questions.items():
        while True:
            try:
                val = int(input(f"   {question}\n   Your answer (0=No / 1=Yes): "))
                if val in [0, 1]:
                    answers[col] = val
                    break
                else:
                    print("   ⚠️  Please enter 0 or 1 only")
            except ValueError:
                print("   ⚠️  Please enter 0 (No) or 1 (Yes)")
    
    # ── SECTION 3: Rating questions ───────────────────────────
    print("\n📋 SECTION 3 — Rate on a scale of 1 to 9")
    print("   1 = Very low / poor     9 = Very high / excellent")
    print()
    
    rating_questions = {
        'Sexual Activity': "15. How would you rate your interest in sexual activity?",
        'Concentration':   "16. How would you rate your ability to concentrate?",
        'Optimisim':       "17. How optimistic are you about your future?"
    }
    
    for col, question in rating_questions.items():
        while True:
            try:
                val = int(input(f"   {question}\n   Your answer (1-9): "))
                if 1 <= val <= 9:
                    answers[col] = val
                    break
                else:
                    print("   ⚠️  Please enter a number between 1 and 9")
            except ValueError:
                print("   ⚠️  Please enter a number between 1 and 9")
    
    # ── GENERATE PREDICTION ───────────────────────────────────
    print("\n" + "="*55)
    print("   Analyzing your responses...")
    print("="*55)
    
    # Convert answers to DataFrame
    input_df = pd.DataFrame([answers])
    input_df = input_df[list(X.columns)]
    
    # Get prediction and probabilities
    prediction_number = model_lr.predict(input_df)[0]
    condition_name    = le.inverse_transform([prediction_number])[0]
    probabilities     = model_lr.predict_proba(input_df)[0]
    confidence        = probabilities[prediction_number] * 100
    
    # ── DISPLAY RESULT ────────────────────────────────────────
    print("\n" + "="*55)
    print("        YOUR MENTAL HEALTH ASSESSMENT")
    print("="*55)
    
    # Condition-specific message
    messages = {
        'Normal':         ("🟢 YOUR MENTAL HEALTH LOOKS GOOD",
                          "No significant signs of mental health conditions\n"
                          "  were detected. Keep maintaining healthy habits!"),
        'Depression':     ("🔵 SIGNS OF DEPRESSION DETECTED",
                          "Your responses suggest signs of depression.\n"
                          "  Please consider speaking with a mental health\n"
                          "  professional for proper evaluation."),
        'Bipolar Type-1': ("🟠 SIGNS OF BIPOLAR TYPE-1 DETECTED",
                          "Your responses suggest signs of Bipolar Type-1.\n"
                          "  This is characterized by extreme mood episodes.\n"
                          "  Please consult a mental health professional."),
        'Bipolar Type-2': ("🟡 SIGNS OF BIPOLAR TYPE-2 DETECTED",
                          "Your responses suggest signs of Bipolar Type-2.\n"
                          "  Characterized by depressive and hypomanic episodes.\n"
                          "  Please consult a mental health professional.")
    }
    
    title, message = messages[condition_name]
    print(f"\n  {title}")
    print(f"\n  {message}")
    print(f"\n  Model confidence : {confidence:.1f}%")
    
    print("\n  Full probability breakdown:")
    for i, cond in enumerate(le.classes_):
        bar    = "█" * int(probabilities[i] * 30)
        marker = " ← predicted" if i == prediction_number else ""
        print(f"    {cond:15s}: {probabilities[i]*100:5.1f}%  {bar}{marker}")
    
    print("\n" + "="*55)
    print("  ⚠️  DISCLAIMER: This is an ML-based screening tool")
    print("  only. It is NOT a medical diagnosis. Always consult")
    print("  a qualified mental health professional.")
    print("="*55)
    
    return condition_name, confidence


# ── RUN THE QUESTIONNAIRE ─────────────────────────────────────
condition, confidence = run_questionnaire()

   MENTAL HEALTH ASSESSMENT QUESTIONNAIRE
Answer each question honestly.
Your answers are used to assess your mental health.

📋 SECTION 1 — How often do you experience these?
   Options: 0=Seldom  1=Sometimes  2=Usually  3=Most-Often



   1. How often do you feel sad or empty?
   Your answer (0/1/2/3):  1
   2. How often do you feel unusually happy or high energy?
   Your answer (0/1/2/3):  1
   3. How often do you feel physically or mentally exhausted?
   Your answer (0/1/2/3):  1
   4. How often do you have trouble sleeping?
   Your answer (0/1/2/3):  1



📋 SECTION 2 — Yes or No questions
   Options: 0=No  1=Yes



   5.  Do you experience sudden extreme mood swings?
   Your answer (0=No / 1=Yes):  1
   6.  Have you had thoughts of self-harm or suicide?
   Your answer (0=No / 1=Yes):  1
   7.  Do you have unusual eating habits or loss of appetite?
   Your answer (0=No / 1=Yes):  1
   8.  Do you generally respect authority figures?
   Your answer (0=No / 1=Yes):  1
   9.  Do you respond aggressively when stressed or upset?
   Your answer (0=No / 1=Yes):  1
   10. Do you tend to ignore problems and move on?
   Your answer (0=No / 1=Yes):  1
   11. Can you easily admit when you make mistakes?
   Your answer (0=No / 1=Yes):  1
   12. Do you overthink situations excessively?
   Your answer (0=No / 1=Yes):  1
   13. Do you experience nervous breakdowns?
   Your answer (0=No / 1=Yes):  1
   14. Do you try to explain your behavior to others?
   Your answer (0=No / 1=Yes):  1



📋 SECTION 3 — Rate on a scale of 1 to 9
   1 = Very low / poor     9 = Very high / excellent



   15. How would you rate your interest in sexual activity?
   Your answer (1-9):  1
   16. How would you rate your ability to concentrate?
   Your answer (1-9):  1
   17. How optimistic are you about your future?
   Your answer (1-9):  1



   Analyzing your responses...

        YOUR MENTAL HEALTH ASSESSMENT

  🟡 SIGNS OF BIPOLAR TYPE-2 DETECTED

  Your responses suggest signs of Bipolar Type-2.
  Characterized by depressive and hypomanic episodes.
  Please consult a mental health professional.

  Model confidence : 99.7%

  Full probability breakdown:
    Bipolar Type-1 :   0.1%  
    Bipolar Type-2 :  99.7%  █████████████████████████████ ← predicted
    Depression     :   0.2%  
    Normal         :   0.0%  

  ⚠️  DISCLAIMER: This is an ML-based screening tool
  only. It is NOT a medical diagnosis. Always consult
  a qualified mental health professional.


## ✅ Final Cell — Jupyter-friendly Interactive Questionnaire

In [60]:
# ============================================================
# UPDATED QUESTIONNAIRE — All 17 questions use same 4 options
# Seldom / Sometimes / Usually / Most-Often
# ============================================================

!pip install ipywidgets -q

import ipywidgets as widgets
from IPython.display import display, clear_output

# All questions use the same 4 options
options = ['Seldom', 'Sometimes', 'Usually', 'Most-Often']

# ── Define all 17 questions ───────────────────────────────────
# Format: (column_name, question_text)
all_questions = [
    # Section 1 — Frequency (direct mapping 0/1/2/3)
    ('Sadness',
     '1.  How often do you feel sad or empty?'),

    ('Euphoric',
     '2.  How often do you feel unusually happy or high energy?'),

    ('Exhausted',
     '3.  How often do you feel physically or mentally exhausted?'),

    ('Sleep dissorder',
     '4.  How often do you have trouble sleeping?'),

    # Section 2 — YES/NO converted to frequency scale
    # Seldom=Never=0, Sometimes=Occasionally=0,
    # Usually=Often=1, Most-Often=Always=1
    ('Mood Swing',
     '5.  How often do you experience sudden extreme mood swings?'),

    ('Suicidal thoughts',
     '6.  How often do you have thoughts of self-harm or suicide?'),

    ('Anorxia',
     '7.  How often do you have unusual eating habits or loss of appetite?'),

    ('Authority Respect',
     '8.  How often do you respect and follow authority figures?'),

    ('Aggressive Response',
     '9.  How often do you respond aggressively when stressed or upset?'),

    ('Ignore & Move-On',
     '10. How often do you ignore problems instead of solving them?'),

    ('Admit Mistakes',
     '11. How often do you admit when you have made a mistake?'),

    ('Overthinking',
     '12. How often do you overthink situations excessively?'),

    ('Nervous Break-down',
     '13. How often do you experience nervous breakdowns?'),

    ('Try-Explanation',
     '14. How often do you try to explain your behavior to others?'),

    # Section 3 — Ratings converted to frequency scale
    # Seldom=low(2), Sometimes=mid-low(4),
    # Usually=mid-high(6), Most-Often=high(8)
    ('Sexual Activity',
     '15. How often do you feel a healthy interest in sexual activity?'),

    ('Concentration',
     '16. How often are you able to concentrate well on tasks?'),

    ('Optimisim',
     '17. How often do you feel optimistic about your future?'),
]

# ── Conversion maps ───────────────────────────────────────────

# For frequency columns — direct 0/1/2/3
freq_map = {
    'Seldom':     0,
    'Sometimes':  1,
    'Usually':    2,
    'Most-Often': 3
}

# For YES/NO columns
# Seldom/Sometimes = No (0) — rarely happens
# Usually/Most-Often = Yes (1) — happens a lot
yes_no_map = {
    'Seldom':     0,
    'Sometimes':  0,
    'Usually':    1,
    'Most-Often': 1
}

# For rating columns (1-9 scale)
# We map to meaningful points on the 1-9 scale
rating_map = {
    'Seldom':     2,   # low
    'Sometimes':  4,   # mid-low
    'Usually':    6,   # mid-high
    'Most-Often': 8    # high
}

# ── Which column belongs to which map ────────────────────────
freq_cols = [
    'Sadness', 'Euphoric', 'Exhausted', 'Sleep dissorder'
]

yes_no_cols = [
    'Mood Swing', 'Suicidal thoughts', 'Anorxia',
    'Authority Respect', 'Aggressive Response', 'Ignore & Move-On',
    'Admit Mistakes', 'Overthinking', 'Nervous Break-down',
    'Try-Explanation'
]

rating_cols = [
    'Sexual Activity', 'Concentration', 'Optimisim'
]

# ── Build the UI ──────────────────────────────────────────────

title = widgets.HTML(
    value="""
    <div style='padding:10px 0'>
        <h2 style='color:#2c3e50;margin:0'>
            🧠 Mental Health Assessment
        </h2>
        <p style='color:#7f8c8d;margin:6px 0'>
            Answer all 17 questions honestly using the same scale:<br>
            <b>Seldom</b> = Rarely &nbsp;|&nbsp;
            <b>Sometimes</b> = Occasionally &nbsp;|&nbsp;
            <b>Usually</b> = Often &nbsp;|&nbsp;
            <b>Most-Often</b> = Almost Always
        </p>
        <hr>
    </div>
    """
)

# Section headers
section_headers = {
    0:  "<p style='background:#e8f4f8;padding:8px 12px;"
        "border-radius:6px;margin:16px 0 8px'>"
        "<b>📋 Section 1 — Emotional symptoms</b></p>",
    4:  "<p style='background:#f0e8f8;padding:8px 12px;"
        "border-radius:6px;margin:16px 0 8px'>"
        "<b>📋 Section 2 — Behavioural symptoms</b></p>",
    14: "<p style='background:#e8f8ee;padding:8px 12px;"
        "border-radius:6px;margin:16px 0 8px'>"
        "<b>📋 Section 3 — Lifestyle &amp; wellbeing</b></p>",
}

# Create one dropdown widget per question
dropdown_widgets = []
for col, question_text in all_questions:
    w = widgets.Dropdown(
        options=options,
        value='Sometimes',       # default value
        description='',
        layout=widgets.Layout(width='260px')
    )
    dropdown_widgets.append((col, question_text, w))

# Build full UI with section headers
ui_items = [title]
for i, (col, question_text, w) in enumerate(dropdown_widgets):
    # Add section header if needed
    if i in section_headers:
        ui_items.append(widgets.HTML(value=section_headers[i]))

    # Question label + dropdown side by side
    label = widgets.HTML(
        value=f"<p style='margin:4px 0;font-size:14px'>"
              f"<b>{question_text}</b></p>"
    )
    row = widgets.VBox(
        [label, w],
        layout=widgets.Layout(margin='0 0 10px 0')
    )
    ui_items.append(row)

# Submit button
submit_btn = widgets.Button(
    description='🔍 Get My Assessment',
    button_style='success',
    layout=widgets.Layout(
        width='220px', height='44px', margin='16px 0'
    )
)

output = widgets.Output()

# ── On button click ───────────────────────────────────────────
def on_submit(button):
    with output:
        clear_output()

        answers = {}

        for col, question_text, w in dropdown_widgets:
            val = w.value   # e.g. 'Usually'

            # Convert using the correct map for this column
            if col in freq_cols:
                answers[col] = freq_map[val]
            elif col in yes_no_cols:
                answers[col] = yes_no_map[val]
            else:
                answers[col] = rating_map[val]   # rating cols

        # Build input DataFrame in correct column order
        input_df = pd.DataFrame([answers])[list(X.columns)]

        # Predict
        pred_num   = model_lr.predict(input_df)[0]
        condition  = le.inverse_transform([pred_num])[0]
        probs      = model_lr.predict_proba(input_df)[0]
        confidence = probs[pred_num] * 100

        # Result styling
        style = {
            'Normal':         ('#27ae60', '🟢',
                               'Your mental health looks good!',
                               'No significant signs detected. '
                               'Keep maintaining healthy habits.'),
            'Depression':     ('#2980b9', '🔵',
                               'Signs of Depression detected.',
                               'Please consider speaking with a '
                               'mental health professional.'),
            'Bipolar Type-1': ('#e67e22', '🟠',
                               'Signs of Bipolar Type-1 detected.',
                               'Characterized by extreme mood episodes. '
                               'Please consult a professional.'),
            'Bipolar Type-2': ('#f39c12', '🟡',
                               'Signs of Bipolar Type-2 detected.',
                               'Characterized by depressive & hypomanic '
                               'episodes. Please consult a professional.')
        }

        color, emoji, headline, detail = style[condition]

        # Probability bars in HTML
        prob_bars = ""
        for i, cond in enumerate(le.classes_):
            pct    = probs[i] * 100
            width  = int(probs[i] * 220)
            marker = " ◀ predicted" if i == pred_num else ""
            c      = color if i == pred_num else "#bdc3c7"
            prob_bars += (
                f"<div style='margin:6px 0;display:flex;"
                f"align-items:center;gap:8px'>"
                f"<span style='width:130px;font-size:13px'>{cond}</span>"
                f"<div style='width:{width}px;height:18px;"
                f"background:{c};border-radius:4px'></div>"
                f"<span style='font-size:13px'>"
                f"{pct:.1f}%{marker}</span>"
                f"</div>"
            )

        # Final result HTML
        result = f"""
        <div style='border:2px solid {color};border-radius:12px;
                    padding:24px;margin-top:12px'>
            <h2 style='color:{color};margin:0 0 8px'>
                {emoji} {headline}
            </h2>
            <p style='color:#555;margin:0 0 12px'>{detail}</p>
            <p style='margin:0 0 16px'>
                <b>Model confidence:</b>
                <span style='color:{color};font-size:18px;
                             font-weight:bold'> {confidence:.1f}%</span>
            </p>
            <hr style='border-color:{color};opacity:0.3;margin:12px 0'>
            <p style='margin:0 0 8px'><b>Probability breakdown:</b></p>
            {prob_bars}
            <hr style='border-color:{color};opacity:0.3;margin:12px 0'>
            <p style='color:#e74c3c;font-size:12px;margin:0'>
                ⚠️ This is an ML screening tool only —
                NOT a medical diagnosis. Always consult
                a qualified mental health professional.
            </p>
        </div>
        """
        display(widgets.HTML(value=result))


submit_btn.on_click(on_submit)

# Display everything
ui_items.append(submit_btn)
ui_items.append(output)
display(widgets.VBox(ui_items))


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
